In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score


train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print(train.shape)
train.head()


print(train.isnull().sum())

print("\nSurvival rate by sex:")
print(train.groupby('Sex')['Survived'].mean())

print("\nSurvival rate by class:")
print(train.groupby('Pclass')['Survived'].mean())

sns.barplot(x='Sex', y='Survived', data=train)
plt.title('Survival rate by gender')
plt.show()


def preprocess(df):
    df = df.copy()
    
    df['Age'].fillna(df['Age'].median(), inplace=True)
    df['Fare'].fillna(df['Fare'].median(), inplace=True)
    df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
    
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
    
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    
    features = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'FamilySize', 'IsAlone']
    return df[features]

X_train = preprocess(train)
y_train = train['Survived']
X_test  = preprocess(test)

print("Training features shape:", X_train.shape)
print("Test features shape:", X_test.shape)


model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
print(f"CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

model.fit(X_train, y_train)
print("Model trained!")


predictions = model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})

submission.to_csv('submission.csv', index=False)

print(f"Saved! Total rows: {len(submission)}")
print(f"Predicted survivors: {predictions.sum()} out of {len(predictions)}")
submission.head(10)